# 01c Ingest AWC METAR

Bronze weather ingestion for EDDF using the official Aviation Weather Center METAR API.

Outputs:
- `adsb_airspace_eddf.brz_weather.metar_raw`
- `adsb_airspace_eddf.obs.ingestion_log`
- `adsb_airspace_eddf.obs.ingestion_partition_log`

Notes:
- The AWC API currently keeps only the previous 15 days of data.
- This notebook uses the METAR product endpoint under `/api/data/metar`.


In [ ]:
from __future__ import annotations

from datetime import datetime, timedelta, timezone
from pathlib import Path
import json
import sys
import uuid
import yaml

if "spark" not in globals():
    raise RuntimeError("This notebook expects a Spark session, for example inside Databricks.")

UTC = timezone.utc
repo_root = Path.cwd().resolve().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.weather import (
    AviationWeatherConfig,
    derive_query_hours,
    fetch_metar_records,
    normalize_metar_records,
    parse_awc_timestamp,
    validate_awc_time_range,
)

with (repo_root / "configs" / "region_config.yaml").open("r", encoding="utf-8") as handle:
    region_config = yaml.safe_load(handle)
with (repo_root / "configs" / "pipeline_config.yaml").open("r", encoding="utf-8") as handle:
    pipeline_config = yaml.safe_load(handle)


def parse_int(raw_value: str) -> int:
    return int(raw_value.strip())


def parse_bool(raw_value: str) -> bool:
    return raw_value.strip().lower() == "true"


def parse_optional_timestamp(raw_value: str):
    text = raw_value.strip()
    if not text:
        return None
    parsed = datetime.fromisoformat(text.replace("Z", "+00:00"))
    if parsed.tzinfo is None:
        return parsed.replace(tzinfo=UTC)
    return parsed.astimezone(UTC)


def utc_now() -> datetime:
    return datetime.now(UTC)


def naive_utc(value: datetime | None) -> datetime | None:
    if value is None:
        return None
    if value.tzinfo is None:
        return value
    return value.astimezone(UTC).replace(tzinfo=None)


def normalize_scalar(value):
    if value is None:
        return None
    if hasattr(value, "to_pydatetime"):
        return value.to_pydatetime()
    return value


def ensure_target_table_exists(table_name: str) -> None:
    if not spark.catalog.tableExists(table_name):
        raise ValueError(f"Missing target table: {table_name}. Run 00_platform_setup_catalog_schema.ipynb first.")


def create_dataframe_for_table(*, target_table: str, rows: list[dict]):
    target_schema = spark.table(target_table).schema
    normalized_rows = [
        {field.name: normalize_scalar(row.get(field.name)) for field in target_schema}
        for row in rows
    ]
    return spark.createDataFrame(normalized_rows, schema=target_schema)


def append_row_to_table(*, target_table: str, payload: dict) -> None:
    create_dataframe_for_table(target_table=target_table, rows=[payload]).write.mode("append").saveAsTable(target_table)


def merge_metar_rows(*, target_table: str, rows: list[dict]) -> int:
    if not rows:
        return 0
    staging_view = f"metar_stage_{uuid.uuid4().hex[:8]}"
    create_dataframe_for_table(target_table=target_table, rows=rows).createOrReplaceTempView(staging_view)
    spark.sql(
        f"""
        MERGE INTO {target_table} AS target
        USING {staging_view} AS source
          ON target.station_id = source.station_id
         AND target.observation_time = source.observation_time
        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
        """
    )
    spark.catalog.dropTempView(staging_view)
    return len(rows)


weather_connection = pipeline_config.get("aviation_weather_connection", {})
weather_ingestion_config = pipeline_config.get("weather_ingestion", {})
default_catalog = pipeline_config.get("catalog_name", "adsb_airspace_eddf")
default_station_id = weather_connection.get("station_id", region_config.get("focus_airport", "EDDF"))
default_start_time = ""
default_end_time = ""
default_lookback_hours = str(weather_ingestion_config.get("default_lookback_hours", 48))
default_dry_run = "false"

catalog = default_catalog
station_id_raw = default_station_id
start_time_raw = default_start_time
end_time_raw = default_end_time
lookback_hours_raw = default_lookback_hours
dry_run_raw = default_dry_run

if "dbutils" in globals():
    dbutils.widgets.text("catalog", default_catalog)
    dbutils.widgets.text("station_id", default_station_id)
    dbutils.widgets.text("start_time", default_start_time)
    dbutils.widgets.text("end_time", default_end_time)
    dbutils.widgets.text("lookback_hours", default_lookback_hours)
    dbutils.widgets.dropdown("dry_run", default_dry_run, ["true", "false"])

    catalog = dbutils.widgets.get("catalog").strip() or default_catalog
    station_id_raw = dbutils.widgets.get("station_id").strip() or default_station_id
    start_time_raw = dbutils.widgets.get("start_time").strip()
    end_time_raw = dbutils.widgets.get("end_time").strip()
    lookback_hours_raw = dbutils.widgets.get("lookback_hours").strip() or default_lookback_hours
    dry_run_raw = dbutils.widgets.get("dry_run").strip() or default_dry_run

selected_station_id = station_id_raw.upper()
lookback_hours = parse_int(lookback_hours_raw)
dry_run = parse_bool(dry_run_raw)
current_utc = utc_now()
selected_start_time = parse_optional_timestamp(start_time_raw)
selected_end_time = parse_optional_timestamp(end_time_raw)
if bool(selected_start_time) != bool(selected_end_time):
    raise ValueError("Provide both start_time and end_time, or leave both blank to use lookback_hours.")
if selected_start_time is None:
    selected_end_time = current_utc
    selected_start_time = current_utc - timedelta(hours=lookback_hours)

max_lookback_days = int(weather_ingestion_config.get("max_lookback_days", 15))
validate_awc_time_range(
    start_time=selected_start_time,
    end_time=selected_end_time,
    now_utc=current_utc,
    max_lookback_days=max_lookback_days,
)
query_hours = derive_query_hours(start_time=selected_start_time, now_utc=current_utc) + 1
run_id = f"weather_ingest_{current_utc.strftime('%Y%m%dT%H%M%SZ')}_{uuid.uuid4().hex[:8]}"

metar_table = f"{catalog}.brz_weather.metar_raw"
ingestion_log_table = f"{catalog}.obs.ingestion_log"
partition_log_table = f"{catalog}.obs.ingestion_partition_log"
for table_name in [metar_table, ingestion_log_table, partition_log_table]:
    ensure_target_table_exists(table_name)

weather_config = AviationWeatherConfig(
    base_url=weather_connection.get("base_url", "https://aviationweather.gov/api/data"),
    station_id=selected_station_id,
    user_agent=weather_connection.get("user_agent", "adsb-airspace-eddf-weather/1.0"),
    max_lookback_days=max_lookback_days,
)


In [ ]:
pipeline_started_at = naive_utc(current_utc)
pipeline_status = "success"
pipeline_error_message = None
rows_read = 0
rows_written = 0
filtered_record_count = 0
oldest_observation_time = None
newest_observation_time = None
partition_value = f"{selected_station_id}|{selected_start_time.isoformat()}|{selected_end_time.isoformat()}"

try:
    fetched_records = fetch_metar_records(config=weather_config, hours=query_hours)
    rows_read = len(fetched_records)

    filtered_records = []
    observation_times = []
    for record in fetched_records:
        observation_time = parse_awc_timestamp(
            record.get("obsTime") or record.get("observationTime") or record.get("reportTime")
        )
        if observation_time is not None:
            observation_times.append(observation_time)
            if selected_start_time <= observation_time <= selected_end_time:
                filtered_records.append(record)

    filtered_record_count = len(filtered_records)
    if observation_times:
        oldest_observation_time = min(observation_times)
        newest_observation_time = max(observation_times)

    if rows_read >= 400 and oldest_observation_time and oldest_observation_time > selected_start_time + timedelta(minutes=5):
        raise ValueError(
            "AWC METAR request appears truncated at the 400-result limit before reaching the requested start_time. "
            "Narrow the requested range or ingest a more recent subset."
        )
    if filtered_record_count == 0:
        raise ValueError("No METAR observations were returned for the requested station and time range.")

    rows = normalize_metar_records(
        records=filtered_records,
        run_id=run_id,
        ingested_at=current_utc,
        source_request_start=selected_start_time,
        source_request_end=selected_end_time,
    )
    if not dry_run:
        rows_written = merge_metar_rows(target_table=metar_table, rows=rows)
except Exception as exc:
    pipeline_status = "failed"
    pipeline_error_message = f"{type(exc).__name__}: {exc}"
    raise
finally:
    append_row_to_table(
        target_table=partition_log_table,
        payload={
            "run_id": run_id,
            "pipeline_name": "01c_ingest_awc_metar",
            "source_object": f"metar:{selected_station_id}",
            "target_table": metar_table,
            "partition_key": "station_and_range",
            "partition_value": partition_value,
            "status": pipeline_status,
            "rows_read": rows_read,
            "rows_written": rows_written,
            "attempt_no": 1,
            "dry_run": dry_run,
            "started_at": pipeline_started_at,
            "completed_at": naive_utc(utc_now()),
            "error_message": pipeline_error_message,
        },
    )
    append_row_to_table(
        target_table=ingestion_log_table,
        payload={
            "run_id": run_id,
            "pipeline_name": "01c_ingest_awc_metar",
            "source_name": "aviationweather",
            "source_object": f"metar:{selected_station_id}",
            "target_table": metar_table,
            "status": pipeline_status,
            "rows_read": rows_read,
            "rows_written": rows_written,
            "planned_partition_count": 1,
            "success_partition_count": 1 if pipeline_status == "success" else 0,
            "failed_partition_count": 0 if pipeline_status == "success" else 1,
            "started_at": pipeline_started_at,
            "completed_at": naive_utc(utc_now()),
            "metadata_json": json.dumps(
                {
                    "station_id": selected_station_id,
                    "query_hours": query_hours,
                    "requested_start_time": selected_start_time.isoformat(),
                    "requested_end_time": selected_end_time.isoformat(),
                    "oldest_observation_time": oldest_observation_time.isoformat() if oldest_observation_time else None,
                    "newest_observation_time": newest_observation_time.isoformat() if newest_observation_time else None,
                    "filtered_record_count": filtered_record_count,
                    "dry_run": dry_run,
                    "base_url": weather_config.base_url,
                },
                sort_keys=True,
            ),
        },
    )

final_summary = {
    "status": pipeline_status,
    "run_id": run_id,
    "station_id": selected_station_id,
    "weather_table": metar_table,
    "requested_start_time": selected_start_time.isoformat(),
    "requested_end_time": selected_end_time.isoformat(),
    "query_hours": query_hours,
    "rows_read": rows_read,
    "rows_filtered": filtered_record_count,
    "rows_written": rows_written,
    "oldest_observation_time": oldest_observation_time.isoformat() if oldest_observation_time else None,
    "newest_observation_time": newest_observation_time.isoformat() if newest_observation_time else None,
    "dry_run": dry_run,
}

final_summary
